# User-Based Collaborative Filtering

Classical user-based neighborhood CF (Resnick et al., 1994; Sarwar et al., 2001). For target user `u` and item `i`, the predicted rating is

$$ \hat{r}_{ui} = \bar{r}_u + \frac{\sum_{v \in N_k(u; i)} \text{sim}(u, v) \, (r_{vi} - \bar{r}_v)}{\sum_{v \in N_k(u; i)} |\text{sim}(u, v)|} $$

where `N_k(u; i)` is the top-`k` users most similar to `u` who **also rated item `i`**, and `sim` is mean-centered cosine similarity (equivalent to Pearson correlation on co-rated items).


In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
import os, random, sys
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

ARTIFACT_DIR    = Path('/content/gdrive/MyDrive/recsys_artifacts')
SPLITS_DIR      = ARTIFACT_DIR / 'splits'
PREDICTIONS_DIR = ARTIFACT_DIR / 'predictions'
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(ARTIFACT_DIR))
from recsys_metrics import build_candidate_lists, evaluate_model

In [3]:
train = pd.read_csv(SPLITS_DIR / 'train.csv')
val   = pd.read_csv(SPLITS_DIR / 'val.csv')
test  = pd.read_csv(SPLITS_DIR / 'test.csv')
trainval = pd.concat([train, val], ignore_index=True)
print(f'train: {len(train):,}  val: {len(val):,}  test: {len(test):,}  trainval: {len(trainval):,}')

train: 99,616  val: 610  test: 610  trainval: 100,226


## Build the sparse user–item matrix and mean-center per user

Pearson correlation between two users equals **cosine similarity of their mean-centered rating vectors restricted to co-rated items**. We approximate this by mean-centering each row at its rated positions (zeros stay zero) and computing cosine similarity. This is the standard scalable implementation; it slightly overweights users with more co-ratings vs strict Pearson but matches what production systems use.

In [4]:
users  = sorted(trainval['userId'].unique())
movies = sorted(trainval['movieId'].unique())
u2i = {u: i for i, u in enumerate(users)}
m2i = {m: i for i, m in enumerate(movies)}

rows = trainval['userId'].map(u2i).to_numpy()
cols = trainval['movieId'].map(m2i).to_numpy()
vals = trainval['rating'].to_numpy(dtype=np.float32)
R = csr_matrix((vals, (rows, cols)), shape=(len(users), len(movies)))

# Per-user mean over rated items (zero entries excluded).
row_sums   = np.asarray(R.sum(axis=1)).flatten()
row_counts = np.asarray((R != 0).sum(axis=1)).flatten()
user_means = np.divide(row_sums, row_counts, out=np.zeros_like(row_sums), where=row_counts > 0)
global_mean = float(trainval['rating'].mean())

# Mean-center at rated positions only. Build a new CSR by subtracting the per-row mean from each nonzero.
R_centered = R.copy().astype(np.float32)
for u in range(R_centered.shape[0]):
    start, end = R_centered.indptr[u], R_centered.indptr[u + 1]
    R_centered.data[start:end] -= user_means[u]

print('R:', R.shape, '  nnz:', R.nnz)

R: (610, 9701)   nnz: 100226


In [5]:
user_sim = cosine_similarity(R_centered, dense_output=True)
np.fill_diagonal(user_sim, 0.0)
print('User-user similarity matrix:', user_sim.shape)

User-user similarity matrix: (610, 610)


## Prediction

In [6]:
TOP_K_NEIGHBORS = 30  # tuned over {10, 20, 30, 50} — 30 was best on val

def predict_rating(user_id, movie_id):
    if user_id not in u2i:
        return global_mean
    u = u2i[user_id]
    if movie_id not in m2i:
        return user_means[u] if row_counts[u] > 0 else global_mean
    i = m2i[movie_id]

    item_col = R.getcol(i).toarray().flatten()
    raters = np.where(item_col > 0)[0]
    if len(raters) == 0:
        return user_means[u] if row_counts[u] > 0 else global_mean

    sims = user_sim[u, raters]
    top_idx = np.argsort(-sims)[:TOP_K_NEIGHBORS]
    top_raters = raters[top_idx]
    top_sims = sims[top_idx]
    if np.abs(top_sims).sum() < 1e-9:
        return user_means[u] if row_counts[u] > 0 else global_mean

    centered = item_col[top_raters] - user_means[top_raters]
    pred = user_means[u] + (top_sims * centered).sum() / np.abs(top_sims).sum()
    return float(np.clip(pred, 0.5, 5.0))

def predict_pointwise(df):
    out = df[['userId', 'movieId', 'rating']].copy()
    out['predicted_rating'] = [predict_rating(u, m) for u, m in zip(df['userId'], df['movieId'])]
    return out

def predict_fn(user_id, movie_ids):
    return np.array([predict_rating(int(user_id), int(m)) for m in movie_ids], dtype=float)

## Evaluate + save

In [7]:
candidates = build_candidate_lists(train, test, num_negatives=99, seed=SEED)
pointwise = predict_pointwise(test)
metrics = evaluate_model(predict_fn, test, candidates, pointwise_predictions=pointwise, k=10, threshold=3.5)
print('=== User-CF — test set ===')
for key in ['rmse', 'mae', 'hr_at_k', 'ndcg_at_k', 'precision_at_k', 'recall_at_k', 'f1_at_k']:
    print(f'  {key:18s} = {metrics[key]:.4f}')

=== User-CF — test set ===
  rmse               = 0.9753
  mae                = 0.7455
  hr_at_k            = 0.0754
  ndcg_at_k          = 0.0380
  precision_at_k     = 0.0067
  recall_at_k        = 0.0672
  f1_at_k            = 0.0122


In [8]:
out_path = PREDICTIONS_DIR / 'user_cf.csv'
pointwise[['userId', 'movieId', 'predicted_rating']].to_csv(out_path, index=False)
print(f'Saved {len(pointwise)} predictions -> {out_path}')

Saved 610 predictions -> /content/gdrive/MyDrive/recsys_artifacts/predictions/user_cf.csv
